<a href="https://colab.research.google.com/github/MihaiDogariu/CV3/blob/main/laborator/CV_3_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introducere în Google Colab
Acest laborator se axează pe explicarea conceptelor ce stau la baza lucrului cu mediul Google Colab și exemplificarea unor funcționalități simple.

##1.1. Ce este Google Colab?

Google Colaboratory sau, pe scurt, **Colab**, este un mediu de lucru ce rulează pe o mașină virtuală în cloud și permite rularea de cod Python direct în browser, cu următoarele caracteristici:
1.   Nu este necesară nicio configurare a mediului de lucru (instalare biblioteci, pachete, utilitare) local;
2.   Acces gratuit la resurse hardware avansate, precum GPU - Graphics Processing Unit (placă grafică);
3.   Posibilitatea de a partaja codul cu oricine.

Fișierele utilizate de Colab mai sunt denumite și "notebook". Colab este o adaptare a proiectului [Jupyter](https://jupyter.org/) la ecosistemul Google. Astfel, un proiect Colab nu este altceva decât un notebook Jupyter cu acces la resursele Google ale utilizatorilor. Aceste notebooks pot fi considerate mini-proiecte, de sine stătătoare. Marele avantaj al acestui mod de lucru este posibilitatea de a accesa resurse hardware online care, altfel, ar avea un cost prohibitiv.

##1.2. Cum funcționează Google Colab?

Conținutul unui notebook Colab este format din celule de tip "Code" sau "Text". Celulele de tipul "Code" conțin secvențe de cod + comentarii, iar celulele "Text" conțin descrieri textuale și vizuale ale diferitelor componente, în format [Markdown](https://colab.research.google.com/notebooks/markdown_guide.ipynb).

Pentru a adăuga celule, se pot folosi butoanele dedicate din partea de sus a notebook-ului:

<div>
  <center>
    <img src="https://drive.google.com/uc?export=view&id=14iBdqO9ZCKSv7m9IqgGeRxoMA_mcHg3x" width="500" class="center">
  </center>
</div>

De asemenea, se poate folosi și meniul "Insert":


<div>
  <center>
    <img src="https://drive.google.com/uc?export=view&id=1TXSId5PdBZA0JmNRZwOIzr9-VBu7DT7O" width="400" class="center">
  </center>
</div>

Întregul document vizualizat acum nu reprezintă o pagină statică de web, ci este un mediu interactiv în care se poate scrie și rula cod Python. De exemplu, celula de mai jos calculează numărul de ore asociat materiei CV 3.

In [ ]:
num_ETC = 5 # puncte credit ETC asociate materiei
hours_per_ETC = 25 # numar de ore asociate fiecărui punct ETC
total_hours_per_subject = num_ETC * hours_per_ETC
print("Materiei CV 3 îi sunt alocate {} ore de studiu pe parcursul semestrului.".format(total_hours_per_subject))

Codul din celula de mai sus poate fi executat selectând celula și apoi apăsând pe butonul "play" din partea stânga-sus a celulei sau cu scurtătura "Command/Ctrl+Enter".
O variabilă care a fost definită într-o celulă ce a fost rulată poate fi utilizată în alte zone ale notebook-ului. De exemplu, în celula de mai jos este utilizată valoarea variabilei "total_hours_per_subject".

Dacă celula de mai sus nu ar fi fost rulată în prealabil, această variabilă nu ar fi fost definită și, deci, nu ar fi putut să fie utilizată.

In [ ]:
CV3_course_hours_per_week = 2
CV3_lab_hours_per_week = 2
weeks_per_semester = 14
individual_work_per_semester = total_hours_per_subject - (CV3_course_hours_per_week + CV3_lab_hours_per_week) * weeks_per_semester
print("Materia CV 3 prevede un total de {} ore de studiu individual pe parcursul semestrului.".format(individual_work_per_semester))

Fiecare notebook Colab este executat într-o instanță virtuală (runtime) ce poate fi găzduită atât local, cât și extern, pe serverele Google. De asemenea, Google pune la dispoziția utilizatorilor o infrastructură hardware considerabilă. Pentru a veni în sprijinul a cât mai mulți utilizatori, aceste resurse sunt eliberate după un anumit timp, pentru a preveni irosirea sau monopolizarea lor.

Pentru asocierea unui accelerator hardware instanței curente de notebook Colab, se poate accesa meniul "Runtime -> Change runtime type -> Hardware accelerator -> GPU".

<div>
  <center>
    <img src="https://drive.google.com/uc?export=view&id=1DlplqRHQRnXjQYspb5cOjWBsMYm-6hjl" width="300" class="center">
  </center>
</div>

<font color='red' size='+2'>
Un GPU poate fi asociat unei instanțe maxim 12 ore, în funcție de disponibilitate și modul de lucru detectat.
</font>

Pentru a testa daca instanța curentă utilizează resurse GPU, puteți rula următorul cod:


In [ ]:
import torch

if torch.cuda.is_available():
    # Obține numele primei plăci grafice (indexul 0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU detectat: {gpu_name}")
    print(f"Număr total de GPU-uri: {torch.cuda.device_count()}")
else:
    print("❌ Nu a fost detectat niciun GPU Accelerator (CUDA). PyTorch rulează pe CPU.")

Îmbunătățirea obținută prin rularea procesării pe GPU în loc de CPU poate fi observată rulând codul de mai jos. Exemplul oferit aplică un strat convoluțional unei imagini aleatoare și compară performanțele obținute rulând procesarea atât pe CPU, cât și pe GPU.

In [ ]:
import torch
import torch.nn as nn
import timeit

# 1. Verificare disponibilitate GPU
if not torch.cuda.is_available():
    print(
        '\n\nThis error most likely means that this notebook is not '
        'configured to use a GPU.  Change this in Notebook Settings via the '
        'command palette (cmd/ctrl-shift-P) or the Edit menu.\n\n')
    raise SystemError('GPU device not found')

# Inițializăm stratul de Convoluție 2D
# În PyTorch: Conv2d(in_channels, out_channels, kernel_size)
# TensorFlow folosea 3 canale de intrare și 32 de ieșire cu kernel de 7
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)

# Definim dispozitivele
device_cpu = torch.device('cpu')
device_gpu = torch.device('cuda:0')

def cpu():
    # PyTorch folosește formatul NCHW: (batch, channels, height, width)
    random_image_cpu = torch.randn(100, 3, 100, 100, device=device_cpu)
    # Trecem modelul pe CPU și rulăm operația
    net_cpu = conv_layer.to(device_cpu)(random_image_cpu)
    return torch.sum(net_cpu)

def gpu():
    # Generăm tensorul direct pe GPU
    random_image_gpu = torch.randn(100, 3, 100, 100, device=device_gpu)
    # Trecem modelul pe GPU și rulăm operația
    net_gpu = conv_layer.to(device_gpu)(random_image_gpu)

    # NOTĂ: PyTorch execută operațiile pe GPU în mod asincron.
    # Pentru o cronometrare corectă, trebuie să forțăm sincronizarea CUDA.
    torch.cuda.synchronize()
    return torch.sum(net_gpu)

# Rulăm fiecare funcție o dată pentru "warm up"
cpu()
gpu()

# Pornim testul de viteză
print('Timp (exprimat in s) necesar pentru a realiza operatia de covolutie '
      'dintre un filtru 32x7x7x3 si un tensor aleator de dimensiune 100x100x100x3'
      '(batch x channel x height x width). Suma dupa 10 iteratii.')

print('CPU (s):')
cpu_time = timeit.timeit('cpu()', number=10, setup="from __main__ import cpu")
print(cpu_time)

print('GPU (s):')
gpu_time = timeit.timeit('gpu()', number=10, setup="from __main__ import gpu")
print(gpu_time)

# Calculăm speedup evitând împărțirea la zero în cazuri extreme
if gpu_time > 0:
    print('Imbunatatire viteza de procesare GPU vs CPU: {}x'.format(int(cpu_time/gpu_time)))
else:
    print('Imbunatatire viteza de procesare GPU vs CPU: Instantaneu (timp GPU ~ 0s)')


##1.3. Integrare cu Google Drive
\[Informații preluate de [aici](https://colab.research.google.com/notebooks/basic_features_overview.ipynb#scrollTo=aro-UJgUQSH1)\]

Colab oferă integrare cu mediul Google Drive în ceea ce privește partajarea, comentariile și colaborarea cu alți utilizatori.


*   Butonul "**Share**" din colțul dreapta-sus vă permite să partajați notebook-ul cu alte persoane;
*   "**File -> Save a copy in Drive**" creează o copie a notebook-ului în Drive-ul propriu;
*   "**File -> Save**" salvează fișierul în Drive;
*   "**File -> Save and pin revision**" salvează fișierul și fixează versiunea curentă astfel încât să nu fie ștearsă din istoricul reviziilor;
*   "**File -> Revision history**" deschide istoricul reviziilor.

Pentru a face comentarii asemănător documentelor din Google Drive (în cazul în care notebook-ul oferă acest drept) există 3 variante:
1.   Se selectează celula și apoi se apasă butonul de comentarii "**Add a comment**"din colțul sus-dreapta al celulei;
1.   Click dreapta pe celulă și apoi "**Add a comment**" din meniul deschis;
1.   Se selectează celula și se folosește scurtătura "**Ctrl+Alt+M**".





## 1.4. Integrare cu GitHub
\[Informații preluate de [aici](https://colab.research.google.com/github/googlecolab/colabtools/blob/master/notebooks/colab-github-demo.ipynb)\]

Colab oferă integrare cu GitHub în ceea ce privește încărcarea notebook-urilor din GitHub, copierea lor în repository personal, participarea la dezvoltarea de proiecte GitHub prin intermediul Colab etc.

*   Încărcarea fișierelor Colab din repository-uri publice din GitHub se face  accesând link-ul GitHub către fișierele respective. Acest link poate fi deschis din interiorul Google Colab accesând meniul "**File -> Open notebook -> tab-ul GitHub -> copiere link în câmpul de search***";
*   De obicei, fișierele Colab din GitHub au o scurtătură atașată care permite accesarea lor directă: ![Exemplu scurătură](https://colab.research.google.com/assets/colab-badge.svg);
*   Încărcarea fișierelor Colab din repository-uri private din GitHub se face urmând următorii pași:

       1. Accesați [http://colab.research.google.com/github](http://colab.research.google.com/github).
       1. Bifați căsuța "Include private repos";
       1. Logați-vă în contul de GitHub și oferiți acces către fișierele private;
       1. Fișierele și repository-urile private vor apărea acum în fereastra Colab.

## 1.5. Alte resurse Colab
* [Colab Features](https://colab.research.google.com/notebooks/basic_features_overview.ipynb#scrollTo=4hfV37gxpP_c)
* [Lucru cu Colab și date statistice](https://colab.research.google.com/notebooks/intro.ipynb#scrollTo=-Rh3-Vt9Nev9)
* [Exemplu de rețea neuronală convoluțională în Keras](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb)
* [Exemplu de rețea perceptron și SVM](https://colab.research.google.com/github/mravanba/comp551-notebooks/blob/master/Perceptron_and_LinearSVM.ipynb#scrollTo=a3MQLll4GKUS)

# 2. Implementarea perceptronului
Să se implementeze o variantă simplă a perceptronului, care modelează funcțiile logice uzuale.

In [ ]:
# Importam bibliotecile necesare (daca au fost deja importate in sesiunea
# curenta atunci nu mai este obligatoriu sa fie incarcate din nou)
import numpy as np
import matplotlib.pyplot as plt
import sys

In [ ]:
input = np.array([(0,0), (0,1), (1,0), (1,1)])
# TODO: Testati diferite functii logice pentru a vedea care dintre ele poate fi
# implementata cu un singur perceptron: OR, AND, XOR.
# Diferenta intre ele consta in schimbarea etichetelor conform tabelei logice.
# Fiecarei pereche de valori din input ii va corespunde eticheta de pe aceeasi
# pozitie din vectorul de etichete.
labels = np.array([0, 1, 1, 1]) # OR

# TODO: Testati moduri diferite de a initializa ponderile: 0 vs aleator
w_init = np.array([0, 0, 0])
# Dimensiunea vectorului de ponderi este cu o unitate mai mare decat dimensiunea
# vectorului de intrare pentru a aloca si polarizarea w_0

In [ ]:
# Functie auxiliara pentru a introduce si intrarea asociata polarizarii: x_0 = 1
def form_x(x):
  return np.concatenate(([1], x))

In [ ]:
def heaviside(x):
  pass # instructiune nula - syntactic placeholder (se va inlocui cu implementarea de mai jos)
  # TODO: implementati functia prag Heaviside

In [ ]:
def forward(x, w):
  pass # instructiune nula - syntactic placeholder (se va inlocui cu implementarea de mai jos)
  # TODO: implementati activarea perceptronului

In [ ]:
num_epochs = 100
# TODO: Testati algoritmul cu diferite valori pentru lr: 0.01, 0.1, 0.5, 1:
lr = 0.01

# Reinitializam ponderile la fiecare rulare a celulei
w = w_init.copy()

for epoch in range(1, num_epochs+1):
  print("Epoca #{}".format(epoch))
  # Retinem valoarea ponderilor de la inceputul epocii
  w_epoch_start = w.copy() # retinem valorile ponderilor de la inceputul epocii
  w_changed = False # flag care ne va spune daca a avut loc vreo schimbare a ponderilor pe parcursul unei epoci
  for i, (input_pair, label) in enumerate(zip(input, labels)):
    # Adaugam x0=1 la vectorul de intrare
    in_values = form_x(input_pair)
    # Calculam activarea perceptronului
    out_value = forward(in_values, w)
    # Actualizam valorile ponderilor
    # TODO
    # Verificam daca a avut loc modificare ponderilor pe parcursul epocii <-> daca algoritmul inca mai invata
    if not (np.array_equal(w, w_epoch_start)):
      w_changed = True
    # Afisam pe ecran valorile conform tabelului din curs, modulul M2, slide 43
    print(in_values, label, out_value, label-out_value, w)
  # Daca valorile w nu se schimba pe parcursul unei epoci atunci putem incheia algoritmul
  if not w_changed:
    print("Nu a fost detectata nicio schimbare a ponderilor la epoca #{}. Algoritmul se poate incheia".format(epoch))
    break

# TODO:
- Completati blocurile de cod astfel incat algoritmul de antrenare a unui perceptron sa functioneze corect
- Realizati testele propuse:
    - diferite valori pentru `lr`: 0.01, 0.1, 0.5, 1
    - moduri diferite de a initializa ponderile: 0 vs aleator
    - diferite functii logice pentru a vedea care dintre ele poate fi implementata cu un singur perceptron: OR, AND, XOR.
- Retineti numarul de epoci necesare pana la convergenta pentru fiecare experiment.
- Realizati un scurt raport in care sa includeti rezultatele experimentelor si concluziile trase (max 1-2 paragrafe) in urma rularii testelor.
- Incarcati raportul respectiv pe Moodle.

